# Test retrieval (vector search -> expand -> preamble/metadata -> group by Article)

Gọi thẳng `rag.retrieval.vector_retriever.retrieve_relevant_clauses()` (không qua LLM evaluate) để
xem TOÀN BỘ kết quả truy hồi: Excerpt nào khớp, điểm số, Clause nào được kéo thêm (cùng Điều, quan
hệ chéo REFERS_TO/DEPENDS_ON/EXCEPTION_TO/REFERS_TO_APPENDIX, USES_TERM/DEFINES), preamble/metadata
hợp đồng có được gắn không. Khác `pipeline_test.ipynb` (chạy hết tới LLM), notebook này dừng ở
retrieval để debug riêng phần này. Output in FULL (không cắt chuỗi).

**Lưu ý API hiện tại**: `retrieve_relevant_clauses(contract_id, query_texts: list[str], ...)` nhận
THẲNG danh sách truy vấn (không nhận dict "item" như bản cũ) - việc phân rã 1 mục checklist thành
truy vấn là trách nhiệm của `checklist.requirement_decomposition.decompose_requirements()` (xem mục 2).

Yêu cầu: contract đã được import + build graph trước đó (contract_id bên dưới phải tồn tại trong
Neo4j - chạy `pipeline_test.ipynb` hoặc `scripts/build_graph.py` trước nếu chưa có).

In [ ]:
import sys
import json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

CONTRACT_ID = 1


def print_full(obj):
    """In FULL, không cắt chuỗi, không rút gọn dict/list (mặc định repr/print của Python/pandas
    hay cắt bớt text dài)."""
    print(json.dumps(obj, ensure_ascii=False, indent=2))


def print_groups(groups):
    """Tóm tắt gọn: mỗi Điều/Phụ lục 1 dòng (article_number, điểm, số Khoản con, độ dài text) -
    tiện xem nhanh KÍCH THƯỚC context sẽ đưa vào LLM mà không cần đọc hết nội dung."""
    total_len = 0
    for g in groups:
        score = f"{g['score']:.4f}" if g["score"] is not None else "-"
        total_len += len(g["text"])
        print(f"{g['article_number']:>8}  score={score:8}  {len(g['clauses']):2d} clause con  {len(g['text']):>7} ky tu")
    print(f"{'':>8}  {'':15} {'':>13}  {total_len:>7} ky tu TONG")

## 1. Retrieval trực tiếp với danh sách truy vấn tự viết tay (không qua LLM phân rã)

Cách nhanh nhất để test 1 truy vấn cụ thể - tự viết `query_texts`, không tốn lệnh LLM phân rã.

In [ ]:
from rag.retrieval.vector_retriever import retrieve_relevant_clauses

query_texts = [
    "Mức phạt vi phạm hợp đồng là bao nhiêu",
    "Tổng mức phạt vi phạm tối đa không vượt quá 8% giá trị hợp đồng",
]

result = retrieve_relevant_clauses(CONTRACT_ID, query_texts)
groups = result["clauses"]
evidences = result["evidences"]
print_groups(groups)

## 1b. Xem "evidences" - cây quan hệ đồ thị đã dẫn tới từng Điều/Khoản khớp trực tiếp

Mỗi phần tử là 1 cây độc lập bắt đầu từ 1 Khoản khớp trực tiếp vector search (root) - "chain"/
"children" là các Khoản được kéo theo qua quan hệ đồ thị thật (REFERS_TO/DEPENDS_ON/EXCEPTION_TO/
REFERS_TO_APPENDIX/DEFINES) hoặc quan hệ suy ra (SAME_ARTICLE - cùng Điều; CONTAINS - gốc Phụ lục
chứa mục con). Có thể lặp: 1 node xuất hiện dưới nhiều root khác nhau nếu có nhiều "cha".

In [ ]:
from checklist.requirement_decomposition import decompose_requirements

item = {
    "id": "adhoc",
    "question": "Người đại diện ký hợp đồng đã xác định đúng theo phân cấp, ủy quyền chưa?",
    "pass_criteria": "Người ký là Giám đốc/Tổng Giám đốc hoặc có giấy ủy quyền hợp lệ.",
    "violation_criteria": "Người ký không có thẩm quyền, không có ủy quyền.",
    "note": "",
}

requirements, usage = decompose_requirements(item)
print("Yêu cầu đã phân rã:")
print_full(requirements)
print("\nUsage:", usage)

query_texts_2 = [q for req in requirements for q in req["queries"]]
groups_2 = retrieve_relevant_clauses(CONTRACT_ID, query_texts_2)["clauses"]
print()
print_groups(groups_2)

## 2. Retrieval đầy đủ pipeline: phân rã 1 mục checklist bằng LLM rồi mới truy hồi

Giống hệt luồng thật (`rag/nodes/generate_queries.py` -> `rag/nodes/retrieval.py`) - tốn 1 lệnh LLM
phân rã trước khi truy hồi.

In [ ]:
from checklist.requirement_decomposition import decompose_requirements

item = {
    "id": "adhoc",
    "question": "Người đại diện ký hợp đồng đã xác định đúng theo phân cấp, ủy quyền chưa?",
    "pass_criteria": "Người ký là Giám đốc/Tổng Giám đốc hoặc có giấy ủy quyền hợp lệ.",
    "violation_criteria": "Người ký không có thẩm quyền, không có ủy quyền.",
    "note": "",
}

requirements, usage = decompose_requirements(item)
print("Yêu cầu đã phân rã:")
print_full(requirements)
print("\nUsage:", usage)

query_texts_2 = [q for req in requirements for q in req["queries"]]
groups_2 = retrieve_relevant_clauses(CONTRACT_ID, query_texts_2)
print()
print_groups(groups_2)

## 3. Xem full text ghép sẵn của từng Điều/Phụ lục (giống context sẽ đưa vào LLM)

In [ ]:
APPENDIX_QUERY = ["Yêu cầu kỹ thuật chi tiết trong phụ lục hợp đồng"]

groups_appendix = retrieve_relevant_clauses(CONTRACT_ID, APPENDIX_QUERY)["clauses"]
print_groups(groups_appendix)

for g in groups_appendix:
    if g.get("is_appendix"):
        print(f"\nPhụ lục {g['article_number']} - {len(g['clauses'])} clause con:")
        for c in g["clauses"]:
            print(f"  {c['number']:12} {len(c['text']):>6} ky tu   {c['title'][:60]}")

## 4. Test riêng Phụ lục - kiểm tra KHÔNG dội cả khối

Sau khi tách Phụ lục thành sub-clause (mỗi mục con 1 Clause riêng) + quan hệ `REFERS_TO_APPENDIX`,
1 truy vấn khớp trực tiếp vào 1 mục con của Phụ lục chỉ nên kéo về ĐÚNG mục con đó (+ node gốc nếu
có Điều khác trỏ chung chung tới cả Phụ lục) - KHÔNG kéo theo mọi mục con anh em (vốn có thể cộng
lại tới hàng chục nghìn ký tự nếu Phụ lục dài).

Đổi `APPENDIX_QUERY` bên dưới cho khớp nội dung Phụ lục thật có trong hợp đồng đang test.

In [ ]:
SAMPLE_ITEMS = [
    {
        "question": "Người đại diện ký hợp đồng đã xác định đúng theo phân cấp, ủy quyền chưa?",
        "pass_criteria": "Người ký là Giám đốc/Tổng Giám đốc hoặc có giấy ủy quyền hợp lệ.",
        "violation_criteria": "Người ký không có thẩm quyền, không có ủy quyền.",
    },
    {
        "question": "Hợp đồng có điều khoản bảo mật thông tin không?",
        "pass_criteria": "",
        "violation_criteria": "",
    },
]

for it in SAMPLE_ITEMS:
    result = retrieve_relevant_clauses(CONTRACT_ID, [it["question"], it["pass_criteria"], it["violation_criteria"]])
    print(f"### Câu hỏi: {it['question']}")
    print_groups(result["clauses"])
    print("=" * 100)
    print()

## 5. Chạy nhiều câu hỏi liên tiếp (batch), in tóm tắt từng câu

In [ ]:
SAMPLE_ITEMS = [
    {
        "question": "Người đại diện ký hợp đồng đã xác định đúng theo phân cấp, ủy quyền chưa?",
        "pass_criteria": "Người ký là Giám đốc/Tổng Giám đốc hoặc có giấy ủy quyền hợp lệ.",
        "violation_criteria": "Người ký không có thẩm quyền, không có ủy quyền.",
    },
    {
        "question": "Hợp đồng có điều khoản bảo mật thông tin không?",
        "pass_criteria": "",
        "violation_criteria": "",
    },
]

for it in SAMPLE_ITEMS:
    result = retrieve_relevant_clauses(CONTRACT_ID, [it["question"], it["pass_criteria"], it["violation_criteria"]])
    print(f"### Câu hỏi: {it['question']}")
    print_groups(result)
    print("=" * 100)
    print()